# DP 2D: стоимость и число маршрутов по сетке

In [ ]:
from pathlib import Path
import csv


def find_data(name):
    import urllib.request
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    url = (
        "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
        "modules/08_09_courier_dp/data/" + name
    )
    dest = Path(name)
    urllib.request.urlretrieve(url, dest)
    return dest.resolve()


def load_coin_cases():
    rows = []
    with find_data("coin_change_cases.csv").open(encoding="utf-8") as file:
        for row in csv.DictReader(file):
            rows.append((
                row["case_id"],
                int(row["amount"]),
                [int(value) for value in row["coins"].split()],
                int(row["expected_min_coins"]),
            ))
    return rows


def load_grid():
    with find_data("route_cost_grid_4x5.csv").open(encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader)
        return [[int(value) for value in row] for row in reader]


COIN_CASES = load_coin_cases()
ROUTE_GRID = load_grid()
assert len(COIN_CASES) == 5
assert len(ROUTE_GRID) == 4 and len(ROUTE_GRID[0]) == 5


## 1. Два параметра состояния

Создайте таблицу того же размера, что и сетка. Ячейка `(row, col)` будет хранить ответ для маршрута до этой точки.

In [ ]:
rows = len(ROUTE_GRID)
cols = len(ROUTE_GRID[0])
dp = None  # TODO: таблица rows × cols, заполненная нулями
assert len(dp) == rows
assert all(len(row) == cols for row in dp)
assert dp[0][0] == 0


## 2. Границы таблицы стоимости

В первую строку можно прийти только слева, в первый столбец — только сверху. Заполните эти базовые состояния.

In [ ]:
dp = [[0] * cols for _ in range(rows)]
# TODO: (0, 0), первая строка, первый столбец
assert dp[0] == [1, 4, 5, 10, 11]
assert [dp[row][0] for row in range(rows)] == [1, 3, 8, 12]


## 3. Минимальная стоимость маршрута

Во внутреннюю ячейку можно прийти сверху или слева. Добавьте стоимость текущей клетки к меньшему из двух предыдущих ответов.

In [ ]:
def min_path_cost(grid):
    # TODO: границы и внутренний переход
    ...


assert min_path_cost(ROUTE_GRID) == 11
assert min_path_cost([[7]]) == 7
assert min_path_cost([[1, 2], [3, 4]]) == 7


## 4. Вернуть всю таблицу

Инженеру нужна диагностика, а не только последняя ячейка. Реализуйте функцию, возвращающую таблицу минимальных стоимостей.

In [ ]:
def min_cost_table(grid):
    # TODO
    ...


costs = min_cost_table(ROUTE_GRID)
assert costs[-1][-1] == 11
assert costs[1][2] == 8
assert all(costs[row][col] >= ROUTE_GRID[row][col] for row in range(rows) for col in range(cols))


## 5. Восстановить маршрут

Идите от правого нижнего угла к соседу с меньшей накопленной стоимостью. Верните координаты от старта до финиша.

In [ ]:
def restore_min_path(grid):
    # TODO: min_cost_table и обратный проход
    ...


path = restore_min_path(ROUTE_GRID)
assert path[0] == (0, 0) and path[-1] == (3, 4)
assert len(path) == 8
assert sum(ROUTE_GRID[row][col] for row, col in path) == 11


## 6. Число маршрутов без препятствий

Теперь значение клетки — количество путей до неё. Стоимости игнорируются; переход складывает число путей сверху и слева.

In [ ]:
def count_paths(rows, cols):
    # TODO
    ...


assert count_paths(4, 5) == 35
assert count_paths(1, 7) == 1
assert count_paths(2, 2) == 2


## 7. Запретные клетки

Запретная клетка получает ноль путей независимо от соседей. Проверьте отдельно случай заблокированного старта.

In [ ]:
def count_paths_with_blocks(rows, cols, blocks):
    # TODO
    ...


assert count_paths_with_blocks(4, 5, {(1, 1), (2, 3)}) == 7
assert count_paths_with_blocks(2, 2, {(0, 0)}) == 0
assert count_paths_with_blocks(2, 2, set()) == 2


## 8. Эксперимент: чувствительность маршрута

Увеличивайте стоимость клетки `(2, 2)` от исходной до 20. Зафиксируйте значения, при которых оптимальный маршрут меняется.

In [ ]:
original_path = restore_min_path(ROUTE_GRID)
changes = []
# TODO: копия сетки для каждого значения 1..20; сравнение пути
assert changes
assert all(1 <= value <= 20 for value in changes)
PATH_NOTE = ""  # TODO: почему локальная правка может сменить весь маршрут, >= 100 символов
assert len(PATH_NOTE) >= 100


## 9. Самостоятельно: максимальная стоимость

Перенесите тот же 2D-шаблон на критерий максимума. Верните число и один маршрут, не изменяя допустимые ходы.

In [ ]:
def max_path_with_route(grid):
    # TODO
    ...


best, path = max_path_with_route(ROUTE_GRID)
assert best == sum(ROUTE_GRID[row][col] for row, col in path)
assert path[0] == (0, 0) and path[-1] == (3, 4)
assert len(path) == 8
